# F6-svd-spectral — Review

Work through this notebook *after* the five lesson sessions and
(ideally) the practice sets.
It is a consolidation tool: a summary table over the unit's six
concepts, the idiom sheet, a self-quiz, and pointers to what to redo.
Quiz answers are collapsed at the end — commit to your answers before
looking.

In [ ]:
import numpy as np

## Concept summary

| Concept | One-line summary | Key fact to retain |
|---|---|---|
| Eigenvalues & eigenvectors | $Sq = \lambda q$, $q \ne 0$: a direction the map only stretches, by signed factor $\lambda$ | $2{\times}2$ by hand: $(S{-}\lambda I)$ rows proportional ⟺ $(a{-}\lambda)(d{-}\lambda) - bc = 0$; $\lambda = 0$ ⟺ singular; eigenvectors are directions — $cq$ (incl. $-q$) equally valid |
| Spectral decomposition | Symmetric $S = Q\Lambda Q^{\mathsf T} = \sum_i \lambda_i q_i q_i^{\mathsf T}$: perpendicular stretch axes | symmetric ⇒ real $\lambda$, orthonormal $Q$ ($Q^{\mathsf T}Q = I$); rank = #nonzero $\lambda$; energy $x^{\mathsf T}Sx \in [\lambda_{\min}, \lambda_{\max}]$; Gram matrices are PSD |
| SVD | Any $W = U\Sigma V^{\mathsf T} = \sum_k \sigma_k u_k v_k^{\mathsf T}$: rotate–stretch–rotate | works for every shape; `svd` returns descending `s` and pre-transposed `Vt`; thin vs full is a real choice (shape table); NumPy's bare default is FULL |
| Singular values | $\sigma_k \ge 0$ descending: the stretch factors $\lVert Wv_k \rVert = \sigma_k$ | rank = #nonzero $\sigma$; **bridge**: thin $U$ ↔ nonzero $\lambda_i = \sigma_i^2$ of $WW^{\mathsf T}$; full $U$ + zero-padded $\lambda$ = complete spectral decomposition; $\sigma(cW) = |c|\sigma(W)$; $\sigma(W^{\mathsf T}) = \sigma(W)$ |
| Frobenius norm | $\lVert A \rVert_F = \sqrt{\sum_{ij} A_{ij}^2}$ — the flattened-vector length | $\lVert W \rVert_F^2 = \sum_i \sigma_i^2$ (component derivation: cross terms die by orthonormality); bare `np.linalg.norm` on 2-D is Frobenius; $\lVert S \rVert_F = \sqrt{\sum \sigma_i^4}$ for $S = WW^{\mathsf T}$ |
| Low-rank approximation | $W_r$ = top-$r$ atoms: `U[:, :r] @ np.diag(s[:r]) @ Vt[:r, :]` | exact error $\lVert W - W_r \rVert_F^2 = \sum_{i>r}\sigma_i^2$ (**dropped** parcels); Eckart–Young: unbeatable; storage $r(n{+}m{+}1)$ vs $nm$; read budgets off the error-vs-$r$ curve |

## The idiom sheet

```python
# eigh on symmetric S: ASCENDING -> the pinned descending reorder
vals_asc, vecs_asc = np.linalg.eigh(S)
vals, vecs = vals_asc[::-1], vecs_asc[:, ::-1]     # values AND columns

# eig on general square M: unsorted, possibly complex
vals, vecs = np.linalg.eig(M)
vals, vecs = vals.real, vecs.real                  # after checking .imag ~ 0
order = np.argsort(vals)[::-1]
vals, vecs = vals[order], vecs[:, order]

# eigen-equation residual (all pairs at once; columns are eigenvectors)
resid = np.abs(S @ vecs - vecs * vals).max()

# spectral reconstruction and atom sum
S_re = vecs @ np.diag(vals) @ vecs.T
atom = vals[i] * (vecs[:, i][:, None] * vecs[:, i][None, :])

# sign-fix pin (distinct, separated eigenvalues only)
j = np.abs(Q).argmax(axis=0)
Q_fixed = Q * np.sign(Q[j, np.arange(Q.shape[1])])

# SVD: thin taught default; s descending; Vt IS V^T
U, s, Vt = np.linalg.svd(W, full_matrices=False)
s_only = np.linalg.svd(W, compute_uv=False)
W_re = U @ np.diag(s) @ Vt

# bridge, full form (n > d): complete spectral decomposition of W W^T
Uf = np.linalg.svd(W, full_matrices=True)[0]
lam_pad = np.concatenate([s**2, np.zeros(n - d)])
S_re = Uf @ np.diag(lam_pad) @ Uf.T

# invariant trio for degenerate spectra (capstone contract)
recon_gap = np.abs(Q @ np.diag(lam) @ Q.T - S).max()
resid = max(np.abs(S @ Q[:, i] - lam[i] * Q[:, i]).max() for i in top)
proj_gap = np.abs(Q[:, :k] @ Q[:, :k].T - U2[:, :k] @ U2[:, :k].T).max()

# Frobenius, truncation, error identity
fro = np.sqrt((A * A).sum())                       # == np.linalg.norm(A)
Wr = U[:, :r] @ np.diag(s[:r]) @ Vt[:r, :]
err = np.sqrt((s[r:]**2).sum())                    # dropped parcels!
rel = err / np.sqrt((s**2).sum())
```

Working habit: after any eigen/SVD unpack, immediately run the
reconstruction gap and (for eigen) the residual — machine-precision or
bust.

## Self-quiz

Fifteen items, all six concepts covered.
Work by hand (calculator-free except where flagged), then check against
the collapsed answers below.

1. Verify or refute: $(3, -1)$ is an eigenvector of
   $\begin{pmatrix} 1 & -3 \\ -2 & 2 \end{pmatrix}$; if it is, name
   the eigenvalue.
2. Find both eigenvalues of $\begin{pmatrix} 6 & 2 \\ 2 & 3
   \end{pmatrix}$ by the dependent-rows route, descending.
3. `np.linalg.eigh(S)` returned `vals = [-4, 1, 7]`.
   Write the reorder idiom's output for `vals` and say what must
   happen to `vecs`.
4. A symmetric $5 \times 5$ matrix has eigenvalues $(9, 4, 4, 1, 0)$.
   (a) Its rank? (b) Which eigenvectors admit entrywise cross-route
   comparison under the sign pin, and which do not?
5. $S$ has spectral data $\lambda = (5, 2)$,
   $q_1 = (1, 0)$, $q_2 = (0, 1)$.
   What is $S$, and what is the energy of the unit direction
   $x = (\tfrac{3}{5}, \tfrac{4}{5})$?
6. Why is every Gram matrix $WW^{\mathsf T}$ PSD?
   One line, the energy argument.
7. Shapes: `np.linalg.svd` on a $(50, 8)$ matrix — give `U`, `s`,
   `Vt` shapes for `full_matrices=False` and for `True`.
8. In the SVD output contract, which array is *already transposed*,
   and which already *sorted*?
9. State the bridge in both forms for $W\ (n, d)$, $n > d$ — one
   sentence each.
10. $W$ has $\sigma = (4, 2, 2, 1)$.
    Compute $\lVert W \rVert_F$ and
    $\lVert S \rVert_F$ for $S = WW^{\mathsf T}$.
11. Same $W$: compute $\lVert W - W_2 \rVert_F$ and the relative
    error (calculator allowed for the final root).
12. Why does the error identity's proof need the tail vectors
    $u_{r+1}, \dots$ to be orthonormal, and where does that
    orthonormality come from?
13. A claimed rank-2 approximation of $W$ from item 10 reports error
    $1.9$.
    Verdict, with the theorem's name.
14. Storage: for a $(300, 60)$ matrix, how many floats does the
    rank-6 factored form need, and what fraction of full storage is
    that?
15. The capstone verified spectral-from-SVD with three invariants
    rather than comparing eigenvector columns.
    Name the trio and the one-word reason columns cannot be compared
    there.

## What to redo, per weak spot

| If you struggled with… | Redo (practice) | Reread (lesson) |
|---|---|---|
| items 1–2 (eigen equation, hand route) | p01, p07, p24 | Session 1 §2–4 |
| item 3 (eigh + reorder idiom) | p09, p14 | Session 2 §2 |
| item 4 (rank from spectrum; sign-pin scope) | p14, p17 | Session 2 §4–5 |
| items 5–6 (spectral form, energy, PSD) | p06, p09, p17 | Session 2 §3, §6 |
| items 7–8 (SVD call, shapes, conventions) | p03, p10, p23 | Session 3 §3–4 |
| item 9 (the bridge, both forms) | p11, p13, p21 | Session 3 §5–6 |
| items 10–11 (norm identities, truncation error) | p02, p04, p05, p08, p12 | Session 4 §1–4; Session 5 §4 |
| item 12 (the proofs) | p15, p16 | Session 4 §2, §4 |
| items 13–14 (Eckart–Young, storage, budgets) | p16, p18, p20 | Session 4 §5–7 |
| item 15 (invariant verification) | p13, p21 | Session 5 §3 |
| normal-form MC decoding | p04 | Sessions 1 §8, 4 §8, 5 §7 |
| integration under pressure | p17, p18, p19, p20, p21, p22 | Session 5 |

## Self-quiz answers

<details><summary><b>Click to reveal (commit to your answers first)</b></summary>

1. $\begin{pmatrix} 1 & -3 \\ -2 & 2 \end{pmatrix}(3, -1)
   = (3 + 3,\; -6 - 2) = (6, -8)$; cross-product
   $6 \cdot (-1) - (-8) \cdot 3 = 18 \ne 0$ — **not** an eigenvector.
2. $(6-\lambda)(3-\lambda) - 4 = \lambda^2 - 9\lambda + 14
   = (\lambda - 7)(\lambda - 2)$: $\lambda = 7, 2$.
3. `vals[::-1] = [7, 1, -4]`; the eigenvector *columns* must be
   reversed with `vecs[:, ::-1]` so pairs stay matched.
4. (a) Rank 4 (four nonzero eigenvalues).
   (b) Entrywise-comparable: the $\lambda = 9$, $\lambda = 1$, and
   $\lambda = 0$ eigenvectors (distinct, separated).
   Not comparable: the two $\lambda = 4$ eigenvectors — a degenerate
   pair, defined only up to rotation; use invariants (their projector).
5. $S = 5\,e_1 e_1^{\mathsf T} + 2\,e_2 e_2^{\mathsf T}
   = \mathrm{diag}(5, 2)$;
   $E = 5 \cdot (3/5)^2 + 2 \cdot (4/5)^2 = 45/25 + 32/25 = 77/25
   = 3.08$.
6. $x^{\mathsf T}WW^{\mathsf T}x = \lVert W^{\mathsf T}x \rVert^2
   \ge 0$ for every $x$, and eigenvalues are energies of unit
   eigenvectors — so none can be negative.
7. Thin: `U (50, 8)`, `s (8,)`, `Vt (8, 8)`.
   Full: `U (50, 50)`, `s (8,)`, `Vt (8, 8)`.
8. `Vt` is already $V^{\mathsf T}$ (rows are right singular vectors);
   `s` is already sorted descending.
9. Thin: the $d$ columns of thin $U$ are eigenvectors of
   $WW^{\mathsf T}$ for its nonzero eigenvalues
   $\lambda_i = \sigma_i^2$ — and nothing more.
   Full: $Q = U_{\text{full}}$ with $\lambda = \sigma^2$ zero-padded
   to length $n$ is the complete spectral decomposition
   $S = Q\Lambda Q^{\mathsf T}$.
10. $\lVert W \rVert_F = \sqrt{16 + 4 + 4 + 1} = \sqrt{25} = 5$;
    $\lVert S \rVert_F = \sqrt{256 + 16 + 16 + 1} = \sqrt{289} = 17$.
11. $\lVert W - W_2 \rVert_F = \sqrt{4 + 1} = \sqrt5 \approx 2.236$;
    relative: $\sqrt5 / 5 \approx 0.447$.
12. Because the identity being applied — $\lVert \cdot \rVert_F^2 =
    \sum \sigma^2$ — holds for a matrix *given in SVD form*, which
    requires orthonormal vector families; the tail vectors inherit
    orthonormality from the full families (removing vectors cannot
    break pairwise orthonormality).
13. The Eckart–Young floor for rank 2 is $\sqrt{4 + 1} = \sqrt5
    \approx 2.236 > 1.9$ — the claim is impossible; reject it.
14. $6 \cdot (300 + 60 + 1) = 2166$ floats of $18000$ — almost exactly
    $12\%$.
15. Reconstruction gap, eigen-equation residuals (top pairs), top-$k$
    projector comparison.
    One word: **degeneracy** (the zero eigenvalue's 160-fold block
    pins down no individual columns).

</details>

---

Scored yourself below ~11/15? Use the redo table above, then retake
the quiz.
At 12+ you are ready for the units that build on this one.

**Going deeper:** the syllabus unit that consumes this one directly is
**`C9-dimensionality-reduction`** — centered data matrices, principal
components as top singular directions, and variance-explained curves;
everything there leans on this unit's bridge and conventions by name.